# 005 · In-Memory Streams with io

> **Standard Library › Files & Paths** · Python 3.14

`io.StringIO` and `io.BytesIO` are file objects that live entirely in memory. Anything that reads or writes a file can work with them instead — which makes them perfect for tests, for building text or binary payloads incrementally, and for capturing output.

**In this notebook you will learn:**

- `StringIO` and `BytesIO` and their .NET counterparts (`StringWriter`, `StringReader`, `MemoryStream`)
- "file-like objects": writing functions that accept *any* stream via duck typing
- `getvalue()` vs `read()`, and the classic `seek(0)` gotcha
- wrapping a `BytesIO` in a `TextIOWrapper` to control encoding
- capturing `print` output with `contextlib.redirect_stdout`
- an overview of the `io` class hierarchy

## .NET mental model

| .NET | Python |
|---|---|
| `StringWriter` / `StringBuilder` | `io.StringIO()` (write, then `getvalue()`) |
| `StringReader` | `io.StringIO("initial text")` (read) |
| `MemoryStream` | `io.BytesIO()` |
| `StreamWriter(memoryStream, Encoding.UTF8)` | `io.TextIOWrapper(bytes_io, encoding="utf-8")` |
| `TextReader` / `TextWriter` / `Stream` (abstract base classes) | `io.TextIOBase` / `io.BufferedIOBase` / `io.IOBase` — but mostly **duck typing** |
| `Console.SetOut(writer)` | `contextlib.redirect_stdout(buffer)` |

The key difference: unlike `StringWriter` (write-only) and `StringReader` (read-only), a single `StringIO` supports reading **and** writing, with one shared position — just like a `MemoryStream`.

## `StringIO` as a writer

```csharp
// C#
using var writer = new StringWriter();
writer.WriteLine("Order report");
writer.Write($"Total: {42}");
string text = writer.ToString();
```

In [ ]:
import io

buf = io.StringIO()
buf.write("Order report\n")
print("Total:", 42, file=buf)        # print() can target any text stream
buf.writelines(["- apple\n", "- pear\n"])

text = buf.getvalue()                # the WHOLE buffer, regardless of position
print(repr(text))

**Tip:** for simply joining many strings, `"".join(parts)` is the idiomatic `StringBuilder`. Reach for `StringIO` when an API expects a *file object* — or when you're writing with `print(..., file=...)`, `csv.writer`, `json.dump`, and so on.

## `StringIO` as a reader

Pass initial text to the constructor and read it exactly like a file — `read()`, `readline()`, or iterate line by line.

```csharp
// C#
using var reader = new StringReader("alice,30\nbob,25\n");
string? line;
while ((line = reader.ReadLine()) is not null) { ... }
```

In [ ]:
source = io.StringIO("alice,30\nbob,25\ncarol,41\n")

print(repr(source.readline()))           # 'alice,30\n'
for line in source:                      # continues from the current position
    name, age = line.rstrip("\n").split(",")
    print(name, int(age))

## The `seek(0)` gotcha

⚠️ **Gotcha:** a stream has one **position**. After writing, the position is at the end — so `read()` returns an empty string. Either `seek(0)` first or use `getvalue()` (which ignores the position).

In [ ]:
buf = io.StringIO()
buf.write("hello world")

print(repr(buf.read()))       # '' — we're at the end
print(buf.tell())             # 11
buf.seek(0)                   # rewind (MemoryStream.Position = 0)
print(repr(buf.read()))       # 'hello world'
print(repr(buf.getvalue()))   # always the full content

⚠️ **Gotcha:** `StringIO("initial")` starts at position **0**, so writing *overwrites* the initial text instead of appending. Seek to the end first (`seek(0, io.SEEK_END)`) if you want to append.

In [ ]:
s = io.StringIO("hello")
s.write("J")
print(s.getvalue())          # Jello — overwritten!

s = io.StringIO("hello")
s.seek(0, io.SEEK_END)
s.write(", world")
print(s.getvalue())          # hello, world

## `BytesIO` — the `MemoryStream`

`BytesIO` is the binary sibling: it reads and writes `bytes`. Use it for in-memory images, zip files, network payloads, or anything else that's "a stream of bytes".

```csharp
// C#
using var ms = new MemoryStream();
ms.Write([0x01, 0x02, 0x03]);
byte[] all = ms.ToArray();
```

In [ ]:
ms = io.BytesIO()
ms.write(b"\x01\x02\x03")
ms.write(bytes([4, 5]))

print(ms.getvalue())          # MemoryStream.ToArray()
print(ms.tell(), len(ms.getvalue()))
ms.seek(1)
print(ms.read(2))             # b'\x02\x03'

Many standard library modules accept a `BytesIO` wherever they want a binary file — for example `zipfile`, so you can build a zip archive without touching the disk (like `new ZipArchive(memoryStream, ZipArchiveMode.Create)`):

In [ ]:
import zipfile

archive = io.BytesIO()
with zipfile.ZipFile(archive, "w") as zf:
    zf.writestr("readme.txt", "hello from memory")
    zf.writestr("data/orders.csv", "id,qty\n1,3\n")

archive.seek(0)                           # rewind before reading!
with zipfile.ZipFile(archive) as zf:
    print(sorted(zf.namelist()))
    print(zf.read("readme.txt").decode("utf-8"))

## File-like objects and duck typing

In C# you'd type a parameter as `TextReader`, `TextWriter` or `Stream`. In Python, functions usually just call `.read()`/`.write()`/iterate — anything with those methods works ("file-like object"). That means **the same function** handles real files, `StringIO`, `sys.stdout`, sockets, and more.

Annotate with `typing.TextIO` / `typing.BinaryIO` (or `io.TextIOBase`) to document the intent *(see Syntax › Classes & OOP › 006 · Abstract Classes and Protocols)*.

In [ ]:
from typing import TextIO

def write_report(orders: list[tuple[str, int]], out: TextIO) -> None:
    for product, qty in orders:
        out.write(f"{product:<8}{qty:>4}\n")

def total_quantity(src: TextIO) -> int:
    return sum(int(line.split()[-1]) for line in src if line.strip())

orders = [("apple", 3), ("pear", 12), ("plum", 7)]
buf = io.StringIO()
write_report(orders, buf)                 # write into memory...
print(buf.getvalue(), end="")

buf.seek(0)
print("total:", total_quantity(buf))      # ...and read it back

The same functions work with a real file — no changes needed:

In [ ]:
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "report.txt"
    with open(path, "w", encoding="utf-8") as f:
        write_report(orders, f)
    with open(path, encoding="utf-8") as f:
        print("total from file:", total_quantity(f))

write_report(orders[:1], sys.stdout)      # or straight to the console

**Tip:** this is why `StringIO` is the go-to for testing I/O code — no temp files, no cleanup. Feed the function a `StringIO` with prepared input and assert on the `getvalue()` of the output buffer.

## Text over bytes: `TextIOWrapper`

What `open()` gives you in text mode is a `TextIOWrapper` (encoding/decoding + newline handling) around a binary buffered stream. You can build the same stack yourself on top of a `BytesIO` — the equivalent of `new StreamWriter(memoryStream, Encoding.Unicode)`.

In [ ]:
raw = io.BytesIO()
writer = io.TextIOWrapper(raw, encoding="utf-16", newline="\n")
writer.write("hi ☕\n")

print(raw.getvalue())            # b'' — still in the wrapper's buffer!
writer.flush()
print(raw.getvalue())            # UTF-16 bytes, with BOM

⚠️ **Gotcha:** closing (or garbage-collecting) the `TextIOWrapper` also closes the underlying `BytesIO`, after which `getvalue()` raises. Call `detach()` to unhook the wrapper and keep the buffer alive — the same issue as `StreamWriter` disposing its `MemoryStream` unless you pass `leaveOpen: true`.

In [ ]:
inner = writer.detach()          # flushes and returns the BytesIO, wrapper becomes unusable
print(inner is raw, raw.closed)  # True False
print(raw.getvalue().decode("utf-16"))

Reading works the same way: wrap incoming bytes (say, from a network response) and iterate decoded lines.

In [ ]:
payload = io.BytesIO("product,price\ncrème brûlée,4.50\n".encode("utf-8"))
for line in io.TextIOWrapper(payload, encoding="utf-8"):
    print(line.rstrip("\n").split(","))

## Capturing output with `redirect_stdout`

`contextlib.redirect_stdout(buf)` temporarily points `sys.stdout` at any text stream — so code that calls plain `print()` writes into your buffer. It's the Python version of `Console.SetOut(new StringWriter())` and is restored automatically. `redirect_stderr` does the same for `sys.stderr`.

In [ ]:
from contextlib import redirect_stdout

def noisy_calculation(n: int) -> int:
    print(f"computing for {n}...")           # code you can't easily change
    return n * n

captured = io.StringIO()
with redirect_stdout(captured):
    result = noisy_calculation(7)

print("result:", result)
print("captured:", repr(captured.getvalue()))

⚠️ **Gotcha:** `redirect_stdout` swaps the global `sys.stdout`, so it affects *all* threads, and it only captures Python-level writes — output from C extensions or child processes goes straight to the real file descriptor.

## The `io` class hierarchy

All file objects derive from abstract base classes in `io` — the counterparts of `Stream`, `TextReader`/`TextWriter`:

```
IOBase                       (close, closed, seekable, __enter__/__exit__, iteration)
├── RawIOBase                unbuffered bytes      → FileIO          (open(..., "rb", buffering=0))
├── BufferedIOBase           buffered bytes        → BufferedReader / BufferedWriter / BufferedRandom
│                                                    BytesIO        (open(..., "rb") etc.)
└── TextIOBase               str, with encoding    → TextIOWrapper  (open(..., "r"))
                                                     StringIO
```

`open()` builds a stack: `TextIOWrapper` → `BufferedReader` → `FileIO` (much like `StreamReader` → `BufferedStream` → `FileStream`).

In [ ]:
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "x.txt"
    p.write_text("x", encoding="utf-8")
    with open(p, encoding="utf-8") as f:
        print(type(f).__name__, "->", type(f.buffer).__name__, "->", type(f.buffer.raw).__name__)

print(isinstance(io.StringIO(), io.TextIOBase), isinstance(io.BytesIO(), io.BufferedIOBase))
print(isinstance(io.StringIO(), io.BufferedIOBase))    # False — text vs binary

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Build text in memory | `StringWriter` / `StringBuilder` | `io.StringIO()` + `.getvalue()` |
| Read text from a string | `StringReader` | `io.StringIO(text)` |
| Bytes in memory | `MemoryStream` | `io.BytesIO()` |
| Get all bytes | `ms.ToArray()` | `bio.getvalue()` |
| Rewind | `ms.Position = 0` | `buf.seek(0)` |
| Text encoding over bytes | `new StreamWriter(ms, enc, leaveOpen: true)` | `io.TextIOWrapper(bio, encoding=...)` + `detach()` |
| Accept any stream | `TextReader` / `Stream` parameter | duck typing, hinted `TextIO` / `BinaryIO` |
| Capture console output | `Console.SetOut(writer)` | `with redirect_stdout(buf):` |

## Exercises

1. Write a function `to_csv_text(rows: list[list[str]]) -> str` that writes the rows with `csv.writer` into a `StringIO` (use `lineterminator="\n"`) and returns the CSV text. Include a value containing a comma to see the quoting.
2. Write `upper_copy(src: TextIO, dst: TextIO) -> int` that copies each line from `src` to `dst` in upper case and returns the number of lines. Test it with two `StringIO` objects.
3. Capture the output of `help(len)` into a string with `redirect_stdout` and print only its first line.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import csv
import io
from contextlib import redirect_stdout
from typing import TextIO

# 1 — csv.writer accepts any object with .write()
def to_csv_text(rows: list[list[str]]) -> str:
    buf = io.StringIO()
    csv.writer(buf, lineterminator="\n").writerows(rows)
    return buf.getvalue()

print(repr(to_csv_text([["id", "name"], ["1", "alice"], ["2", "bob, jr."]])))

# 2 — duck-typed streams
def upper_copy(src: TextIO, dst: TextIO) -> int:
    count = 0
    for line in src:
        dst.write(line.upper())
        count += 1
    return count

src, dst = io.StringIO("one\ntwo\nthree\n"), io.StringIO()
print(upper_copy(src, dst), repr(dst.getvalue()))

# 3 — help() prints to stdout
buf = io.StringIO()
with redirect_stdout(buf):
    help(len)
print(buf.getvalue().splitlines()[0])
```

</details>